# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.0 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile, glob, sys,math, random, collections, csv
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict


In [5]:
TASK_ID='task028'; CH=10; H=W=30
ROOT=Path(COMPETITION)
TASK_PATH=ROOT/f'{TASK_ID}.json'
if not TASK_PATH.exists():
    TASK_PATH=Path(LOCAL_DATA)/f'{TASK_ID}.json'
OUT_DIR=Path.cwd()
ONNX_PATH=OUT_DIR/f'{TASK_ID}_static_graph.onnx'
ZIP_PATH=OUT_DIR/f'{TASK_ID}_static_graph_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'
HEALTH_PATH=OUT_DIR/f'{TASK_ID}_onnx_health.json'

In [6]:

def active_mask(x):
    return (x.sum(1, keepdim=True) > 0).float()

def grid_to_tensor(grid):
    g=np.array(grid,dtype=np.int64)
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for r in range(g.shape[0]):
        for c in range(g.shape[1]):
            v=int(g[r,c])
            if 0 <= v < CH:
                x[0,v,r,c]=1.0
    return x

def tensor_to_grid(y, shape):
    y=np.asarray(y)
    if y.ndim==4:
        y=y[0]
    pred=y.argmax(axis=0).astype(np.int64)
    return pred[:shape[0], :shape[1]]

def exact_eval(sess, examples):
    exact=0; first=None
    name=sess.get_inputs()[0].name
    for i,ex in enumerate(examples):
        out=np.array(ex['output'],dtype=np.int64)
        y=sess.run(None,{name:grid_to_tensor(ex['input'])})[0]
        pred=tensor_to_grid(y, out.shape)
        ok=np.array_equal(pred,out)
        exact += int(ok)
        if not ok and first is None:
            first={'idx':i,'wrong_pixels':int((pred!=out).sum())}
    return {'exact':exact,'total':len(examples),'first_wrong':first}

def op_counts(model):
    return dict(collections.Counter(n.op_type for n in model.graph.node))

def onnx_shape(value_info):
    dims=[]
    for d in value_info.type.tensor_type.shape.dim:
        dims.append(int(d.dim_value) if d.dim_value else None)
    return dims


class ARCModel(nn.Module):
    """Neural-symbolic tensor model: infer top/bottom marker colors and draw two fixed rectangular frames."""
    def __init__(self):
        super().__init__()
        tm=torch.zeros(1,1,30,30); bm=torch.zeros(1,1,30,30)
        for r in [0,2]:
            tm[:,:,r,0:10]=1
        for r in [1,3,4]:
            tm[:,:,r,0]=1; tm[:,:,r,9]=1
        for r in [7,9]:
            bm[:,:,r,0:10]=1
        for r in [5,6,8]:
            bm[:,:,r,0]=1; bm[:,:,r,9]=1
        self.register_buffer('top_mask',tm)
        self.register_buffer('bottom_mask',bm)
    def forward(self,x):
        active=active_mask(x)
        out=[torch.zeros_like(active) for _ in range(10)]
        top_scores=[]; bottom_scores=[]
        for c in range(1,10):
            top_scores.append(x[:,c:c+1,2:3,:].sum((2,3),keepdim=True))
            bottom_scores.append(x[:,c:c+1,7:8,:].sum((2,3),keepdim=True))
        top_max=top_scores[0]; bottom_max=bottom_scores[0]
        for s in top_scores[1:]:
            top_max=torch.maximum(top_max,s)
        for s in bottom_scores[1:]:
            bottom_max=torch.maximum(bottom_max,s)
        nz=torch.zeros_like(active)
        for i,c in enumerate(range(1,10)):
            st=((top_scores[i]==top_max)&(top_max>0)).float()
            sb=((bottom_scores[i]==bottom_max)&(bottom_max>0)).float()
            out[c]=torch.maximum(st*self.top_mask*active, sb*self.bottom_mask*active)
            nz=torch.maximum(nz,out[c])
        out[0]=active*(1-nz)
        return torch.cat(out,dim=1)


In [7]:
with open(TASK_PATH) as f:
    task=json.load(f)
print({k:len(task.get(k,[])) for k in ['train','test','arc-gen']})

{'train': 2, 'test': 1, 'arc-gen': 262}


In [8]:
model=ARCModel().eval()
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)

torch.onnx.export(model,dummy,str(ONNX_PATH),
                  input_names=['input'],output_names=['output'],
                  opset_version=13,dynamic_axes=None,
                  do_constant_folding=True,dynamo=False)

m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m)
ops=op_counts(m)
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
health={
    'task_id':TASK_ID,
    'input_shape':onnx_shape(m.graph.input[0]),
    'output_shape':onnx_shape(m.graph.output[0]),
    'size_bytes':ONNX_PATH.stat().st_size,
    'op_counts':ops,
    'forbidden_ops_present':sorted(forbidden.intersection(ops)),
    'accuracy':{}
}
print(health)

/tmp/ipykernel_16/1900624138.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),


{'task_id': 'task028', 'input_shape': [1, 10, 30, 30], 'output_shape': [1, 10, 30, 30], 'size_bytes': 35166, 'op_counts': {'Constant': 115, 'ReduceSum': 19, 'Greater': 3, 'Cast': 19, 'Slice': 27, 'Max': 34, 'Equal': 18, 'And': 18, 'Mul': 37, 'Sub': 1, 'Concat': 1}, 'forbidden_ops_present': [], 'accuracy': {}}


In [9]:
sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])

for split in ['train','test']:
    res=exact_eval(sess,task.get(split,[]))
    health['accuracy'][split]=f"{res['exact']}/{res['total']}"
    if res['first_wrong'] is not None:
        raise AssertionError((split,res))

ag=task.get('arc-gen',[])
cut=int(round(0.4*len(ag)))
fit=exact_eval(sess,ag[:cut])
hold=exact_eval(sess,ag[cut:])
health['accuracy']['arc_gen_fit_40pct']=f"{fit['exact']}/{fit['total']}"
health['accuracy']['arc_gen_holdout_60pct']=f"{hold['exact']}/{hold['total']}"
if fit['first_wrong'] is not None:
    raise AssertionError(('arc-gen fit split',fit))
if hold['first_wrong'] is not None:
    raise AssertionError(('arc-gen holdout split',hold))

assert health['input_shape']==[1,10,30,30]
assert health['output_shape']==[1,10,30,30]
assert health['size_bytes'] < 1_400_000
assert not health['forbidden_ops_present']
with open(HEALTH_PATH,'w') as f:
    json.dump(health,f,indent=2)
health

{'task_id': 'task028',
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'size_bytes': 35166,
 'op_counts': {'Constant': 115,
  'ReduceSum': 19,
  'Greater': 3,
  'Cast': 19,
  'Slice': 27,
  'Max': 34,
  'Equal': 18,
  'And': 18,
  'Mul': 37,
  'Sub': 1,
  'Concat': 1},
 'forbidden_ops_present': [],
 'accuracy': {'train': '2/2',
  'test': '1/1',
  'arc_gen_fit_40pct': '105/105',
  'arc_gen_holdout_60pct': '157/157'}}

In [10]:
for zp in [ZIP_PATH, GENERIC_ZIP]:
    if zp.exists():
        zp.unlink()
    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
print('wrote', ZIP_PATH, 'and', GENERIC_ZIP)
print('zip contents:', zipfile.ZipFile(GENERIC_ZIP).namelist())

wrote /kaggle/working/task028_static_graph_submission.zip and /kaggle/working/submission.zip
zip contents: ['task028.onnx']
